In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt
from umap import UMAP
import sklearn
import seaborn as sns
from COSMOS import cosmos
from COSMOS.pyWNN import pyWNN
import h5py
import warnings
warnings.filterwarnings('ignore')
random_seed = 20

import os

In [ ]:
# Set the directory for the datasets and the output directory
data_dir = '/data/hulei/STmultiVerse/STmultiVerse_reproducibility/Fig2_Benchmark/Original_Simulated_Data'
output_dir = '/data/hulei/STmultiVerse/STmultiVerse_reproducibility/Fig2_Benchmark/Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for i in range(1, 6):
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad data.")
    # Read the RNA and ATAC datasets
    adata_rna = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_atac = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_atac.h5ad')
    sc.pp.highly_variable_genes(adata_atac, n_top_genes=50000)
    adata_atac = adata_atac[:, adata_atac.var['highly_variable'] == True]
    
    ## COSMOS training
    cosmos_comb = cosmos.Cosmos(adata1=adata_rna,adata2=adata_atac)
    cosmos_comb.preprocessing_data(n_neighbors = 10)
    cosmos_comb.train(spatial_regularization_strength=0.01, z_dim=50,
             lr=1e-3, wnn_epoch = 500, total_epoch=1000, max_patience_bef=10, max_patience_aft=30, min_stop=200,
             random_seed=random_seed, gpu=0, regularization_acceleration=True, edge_subset_sz=1000000)
    weights = cosmos_comb.weights
    df_embedding = pd.DataFrame(cosmos_comb.embedding)

    adata_rna.obsm['cosmos'] = df_embedding.values
    ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['cosmos'].shape[1],
                    use_rep='cosmos')
    ov.utils.cluster(adata_rna, method='leiden', resolution=0.1)

    sc.pl.spatial(adata_rna,
              color=['leiden','cell_type'], colorbar_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=12,
             ) 
    
    # Save the processed dataset
    output_path = f'{output_dir}/Simulated_Dataset_{i}/cosmos_multiomics.h5ad'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    adata_rna.write_h5ad(output_path, compression='gzip')

In [ ]:
!pip list